# Dependencies

In [ ]:
!python -m pip install nltk

# Fix data syntax manually

### Find errors

In [53]:
from nltk.sem import Expression
import re

def translate_fol(fol_text: str):
    # '-' --> '_'
    fol_text = re.sub(r'(?<=[a-zA-Z0-9])-(?=[a-zA-Z0-9])', '_', fol_text)

    replacements = {
        '∀': ' all ', 
        '∃': ' exists ',
        '∧': '&', 
        '∨': '|',
        '⊕': '^',
        '¬': '-',
        '→': '->', 
        '—>': '->',
        '⟷': '<->',
        '↔': ' <-> '
    }
    for k, v in replacements.items():
        fol_text = fol_text.replace(k, v)

    # Add dot: "all x (P(x))" --> "all x. (P(x))"
    fol_text = re.sub(r'(all|exists)\s+([a-z0-9]+)\s*', r'\1 \2. ', fol_text)

    # Fix prover9 constants name (eg: "yuri" -> "c_yuri")
    words = re.findall(r'\b[a-z][a-zA-Z0-9_]*\b', fol_text)
    reserved_words = {'all', 'exists', 'u', 'v', 'w', 'x', 'y', 'z'}
    
    for w in set(words):
        if w not in reserved_words:
            fol_text = re.sub(fr'\b{w}\b', f'c_{w}', fol_text)
    return fol_text

In [ ]:
import json

file_path = 'folio_test.json'
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

error_ids = set()
for d in data:
    fol_list = d["fol"].split('\n')
    for fol in fol_list:
        try:
            read_expr = Expression.fromstring(translate_fol(fol))
        except Exception as e:
            id = d["story_id"]
            print(f"{id}: {e}")
            error_ids.add(id)

print(sorted(error_ids))

[]


### Fix files

In [ ]:
import re
import json

def fix_logic_errors(text):
    if not isinstance(text, str) or not text.strip(): return text

    # c_mr_Andmrs_Smith -> c_mr_AndMrs_Smith
    text = re.sub(r'c_[a-zA-Z0-9\._]+', lambda m: m.group(0).replace('.', '_'), text)
    # xm c_bolt -> c_xm_bolt
    text = text.replace('c_xm c_bolt', 'c_xm_bolt')
    # c_l-2021 -> c_l_2021
    text = re.sub(r'([a-zA-Z0-9_]+)-([a-zA-Z0-9_]+)', r'\1_\2', text)

    # exists x y.  -> exists x. exists y.
    text = re.sub(r'exists\s+c_([x-z])exists\.\s*([x-z])', r'exists \1. exists \2. ', text)
    # exists c_t. -> exists t.
    text = re.sub(r'exists\s+c_([a-z])(\s|\.)', r'exists \1 ', text)
    # exists x. . -> exists x.
    text = text.replace('. .', '.')
    # exists (Predicate...) -> exists x. (Predicate...)
    text = re.sub(r'exists\s*\(', 'exists x. (', text)

    # Cost(gre, 205 & ... -> Cost(c_gre, c_205) & ...
    text = re.sub(r'([A-Z][a-zA-Z0-9_]*\([^)]+?)(?=\s*[&|^\-><])', r'\1)', text)
    
    # Delete "Never ..." in fol
    if "Never:" in text:
        text = text.split("Never:")[0]
    # Delete '.' at the end of each fol
    text = text.strip()
    if text.endswith('.'): text = text[:-1]
    # 
    text = text.replace(',)', ')')

    # Fix ( )
    for _ in range(3): 
        text = re.sub(r'\)\)+(\s*(?:->|&|\||\^))', r')\1', text)
        
    open_p = text.count('(')
    close_p = text.count(')')
    
    if open_p > close_p:
        text += ')' * (open_p - close_p)
    elif close_p > open_p:
        while text.endswith(')') and text.count(')') > text.count('('):
            text = text[:-1]

    text = text.replace('((x)', '(x)').replace('((y)', '(y)')

    return text.strip()

def fix_file(file_path, delete_stories = []):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    data = [d for d in data if d["story_id"] not in delete_stories]
    for d in data:
        fol_list = d["fol"].split('\n')
        fixed_fol_list = [fix_logic_errors(s) for s in fol_list]
        d["fol"] = '\n'.join(fixed_fol_list)
    
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)


In [ ]:
fix_file('folio_train.json', [11, 103, 155, 341, 373, 376, 401, 409, 410, 411, 445, 447])
fix_file('folio_valid.json', [368])
fix_file('folio_test.json', [380, 318])

# Statistics

In [7]:
splits = ["train", "valid", "test"]
for spl in splits:
    with open(f"folio_{spl}.json", 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"{spl}'s count:", len(data))

train's count: 955
valid's count: 97
test's count: 97
